# IMPORT LIBRARIES

In [28]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import re
import string
import os

# DATA PREPARATION


In [29]:
def load_dataset(file_path):
    """
    Membaca file imdb_labelled.txt
    Format tiap baris: <teks>\t<label>
    """
    texts = []
    labels = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # Pecah berdasarkan tab
            line_split = line.strip().split('\t')
            if len(line_split) == 2:
                text, label = line_split
                texts.append(text)
                labels.append(int(label))
    return texts, labels

In [30]:
def text_cleaning(text):
    """
    Membersihkan teks sederhana: lower, hapus angka & tanda baca, dsb.
    """
    text = text.lower()
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)  # hapus tanda baca
    text = re.sub(r'\d+', '', text)  # hapus angka
    text = re.sub(r'\s+', ' ', text)  # hapus spasi berlebih
    text = text.strip()
    return text

In [31]:
def build_vocab(cleaned_texts, min_freq=1):
    """
    Membuat word2idx sederhana berdasarkan frekuensi kemunculan kata.
    min_freq = minimal frekuensi agar kata dimasukkan ke vocab
    """
    freq_dict = {}
    for text in cleaned_texts:
        for word in text.split():
            freq_dict[word] = freq_dict.get(word, 0) + 1

    # sort by frequency
    sorted_words = sorted(freq_dict.items(), key=lambda x: x[1], reverse=True)

    # Buat word2idx, sisipkan token khusus <PAD> dan <UNK>
    word2idx = {'<PAD>':0, '<UNK>':1}
    idx = 2
    for word, freq in sorted_words:
        if freq >= min_freq:
            word2idx[word] = idx
            idx += 1
    return word2idx

In [32]:
def encode_text(text, word2idx, max_len=50):
    """
    Mengubah teks menjadi list of token-id (integer),
    dan memotong/padding sampai max_len.
    """
    tokens = text.split()
    encoded = []
    for token in tokens:
        encoded.append(word2idx.get(token, word2idx['<UNK>']))
    # potong jika panjang > max_len
    encoded = encoded[:max_len]
    # padding jika panjang < max_len
    if len(encoded) < max_len:
        encoded += [word2idx['<PAD>']] * (max_len - len(encoded))
    return encoded

In [33]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len=50):
        self.texts = texts
        self.labels = labels
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoded_text = encode_text(text, self.word2idx, self.max_len)
        return torch.LongTensor(encoded_text), torch.LongTensor([label])  # (seq), (label)


# MODEL DEFINITION


In [34]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128, num_layers=1,
                 pooling='max', num_classes=2):
        super(RNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # RNN (vanilla RNN); jika ingin LSTM/GRU ganti di sini
        self.rnn = nn.RNN(input_size=embed_dim,
                          hidden_size=hidden_size,
                          num_layers=num_layers,
                          batch_first=True,
                          nonlinearity='tanh',
                          dropout=0.0,
                          bidirectional=False)

        self.pooling = pooling  # 'max' or 'avg'
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        rnn_out, h_n = self.rnn(embedded)  # (batch_size, seq_len, hidden_size)

        if self.pooling == 'max':
            # MaxPooling over seq_len
            pooled, _ = torch.max(rnn_out, dim=1)  # (batch_size, hidden_size)
        else:
            # AvgPooling
            pooled = torch.mean(rnn_out, dim=1)    # (batch_size, hidden_size)

        logits = self.fc(pooled)  # (batch_size, num_classes)
        return logits

# TRAINING & EVALUATION UTILS


In [35]:
def calculate_accuracy(y_pred, y_true):
    """
    Hitung akurasi antara prediksi dan label sebenarnya.
    """
    predicted = torch.argmax(y_pred, dim=1)
    correct = (predicted == y_true.view(-1)).sum().item()
    total = y_true.size(0)
    return correct / total

In [36]:
def train_one_epoch(model, dataloader, optimizer, criterion, device='cpu'):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    for x_batch, y_batch in dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch.view(-1))
        loss.backward()
        optimizer.step()

        acc = calculate_accuracy(outputs, y_batch)
        running_loss += loss.item() * x_batch.size(0)
        running_acc  += acc * x_batch.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc

In [37]:
def validate_one_epoch(model, dataloader, criterion, device='cpu'):
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch.view(-1))

            acc = calculate_accuracy(outputs, y_batch)
            running_loss += loss.item() * x_batch.size(0)
            running_acc  += acc * x_batch.size(0)
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc

# EARLY STOPPING CLASS


In [38]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0.0):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.delta = delta

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# MAIN TRAINING FUNCTION


In [39]:
def train_and_evaluate(model,
                       train_loader,
                       val_loader,
                       optimizer,
                       scheduler,
                       criterion,
                       epochs=10,
                       patience=5,
                       device='cpu'):

    early_stopper = EarlyStopping(patience=patience)

    best_val_loss = float('inf')
    best_model_state = None

    for epoch in range(1, epochs+1):
        # Training
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        # Validation
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # Scheduler step (misalnya StepLR atau lainnya)
        if scheduler is not None:
            scheduler.step(val_loss)  # Jika pakai ReduceLROnPlateau, butuh param (val_loss)

        print(f"Epoch [{epoch}/{epochs}] | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        # Cek apakah ini model terbaik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()

        # Early Stopping
        early_stopper(val_loss)
        if early_stopper.early_stop:
            print("Early stopping triggered!")
            break

    # Kembalikan state terbaik
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model

# EXPERIMENT CONFIG


In [40]:
from google.colab import drive
import pandas as pd
from google.colab import files
drive.mount('/content/drive')

FILE_PATH = "/content/drive/MyDrive/Machine Learning/imdb_labelled.txt"  # sesuaikan dengan path dataset
BATCH_SIZE = 32
MAX_LEN = 50
EMBED_DIM = 128
DEVICE = 'cpu'  # jika ingin pakai GPU: DEVICE = 'cuda' (jika tersedia)

# Hyper-parameters yang mau kita bandingkan
HIDDEN_SIZES = [64, 128]
POOLINGS = ['max', 'avg']
OPTIMIZERS = ['SGD', 'RMSProp', 'Adam']
EPOCHS_LIST = [5, 50, 100, 250, 350]

# Callback/EarlyStop param
PATIENCE = 5

results = []

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# MAIN EXPERIMENT SCRIPT


In [41]:
def main_experiment_with_logging():
    # 1. Load data
    texts, labels = load_dataset(FILE_PATH)
    # 2. Bersihkan teks
    cleaned_texts = [text_cleaning(t) for t in texts]
    # 3. Build vocab
    word2idx = build_vocab(cleaned_texts, min_freq=1)
    vocab_size = len(word2idx)

    # 4. Train-Val split
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        cleaned_texts, labels, test_size=0.2, random_state=42
    )

    # 5. Buat dataset & dataloader
    train_dataset = IMDBDataset(train_texts, train_labels, word2idx, max_len=MAX_LEN)
    val_dataset   = IMDBDataset(val_texts,   val_labels,   word2idx, max_len=MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

    # 6. Criterion
    criterion = nn.CrossEntropyLoss()

    # 7. Kita akan menyimpan hasil di list "results"
    results = []  # akan menampung dict di setiap kombinasi

    # 8. Looping sesuai settingan experiment
    for hidden_size in HIDDEN_SIZES:
        for pooling in POOLINGS:
            for opt_name in OPTIMIZERS:
                for epochs in EPOCHS_LIST:
                    print("================================================")
                    print(f"[INFO] Training with hidden_size={hidden_size}, pooling={pooling}, "
                          f"optimizer={opt_name}, epochs={epochs}")

                    # Buat model baru
                    model = RNNClassifier(
                        vocab_size=vocab_size,
                        embed_dim=EMBED_DIM,
                        hidden_size=hidden_size,
                        num_layers=1,  # ubah jika mau lebih deep
                        pooling=pooling,
                        num_classes=2
                    ).to(DEVICE)

                    # Buat optimizer
                    if opt_name == 'SGD':
                        optimizer = optim.SGD(model.parameters(), lr=0.01)
                    elif opt_name == 'RMSProp':
                        optimizer = optim.RMSprop(model.parameters(), lr=0.001)
                    else:  # Adam
                        optimizer = optim.Adam(model.parameters(), lr=0.001)

                    # Scheduler (contoh: ReduceLROnPlateau)
                    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

                    # Train & Evaluate (dengan EarlyStopping)
                    trained_model = train_and_evaluate(
                        model,
                        train_loader,
                        val_loader,
                        optimizer,
                        scheduler,
                        criterion,
                        epochs=epochs,
                        patience=PATIENCE,
                        device=DEVICE
                    )

                    # Setelah training selesai (atau early stop),
                    # kita ukur final performance di train set & val set
                    final_train_loss, final_train_acc = validate_one_epoch(trained_model, train_loader, criterion, DEVICE)
                    final_val_loss,   final_val_acc   = validate_one_epoch(trained_model, val_loader,   criterion, DEVICE)

                    # Simpan ke "results"
                    results.append({
                        "hidden_size": hidden_size,
                        "pooling": pooling,
                        "optimizer": opt_name,
                        "epochs": epochs,
                        "final_train_loss": final_train_loss,
                        "final_train_acc": final_train_acc,
                        "final_val_loss": final_val_loss,
                        "final_val_acc": final_val_acc
                    })

                    print("================================================\n")

    # 9. Buat DataFrame dari results
    df_results = pd.DataFrame(results)
    # 10. Simpan ke CSV
    df_results.to_csv('experiment_results.csv', index=False)
    print("Hasil eksperimen telah disimpan ke 'experiment_results.csv'.")

    # 11. Download CSV ke lokal (khusus untuk Colab)
    files.download('experiment_results.csv')
    print("File CSV didownload ke komputer lokal Anda.")

# Jalankan experiment
if __name__ == "__main__":
    main_experiment_with_logging()

[INFO] Training with hidden_size=64, pooling=max, optimizer=SGD, epochs=5
Epoch [1/5] | Train Loss: 0.6984, Train Acc: 0.5138 | Val Loss: 0.7078, Val Acc: 0.4550
Epoch [2/5] | Train Loss: 0.6964, Train Acc: 0.4975 | Val Loss: 0.7047, Val Acc: 0.4750
Epoch [3/5] | Train Loss: 0.6922, Train Acc: 0.5275 | Val Loss: 0.6948, Val Acc: 0.5300
Epoch [4/5] | Train Loss: 0.6913, Train Acc: 0.5212 | Val Loss: 0.7096, Val Acc: 0.4550
Epoch [5/5] | Train Loss: 0.6919, Train Acc: 0.5050 | Val Loss: 0.7031, Val Acc: 0.4850

[INFO] Training with hidden_size=64, pooling=max, optimizer=SGD, epochs=50
Epoch [1/50] | Train Loss: 0.7021, Train Acc: 0.4875 | Val Loss: 0.7076, Val Acc: 0.4550
Epoch [2/50] | Train Loss: 0.7003, Train Acc: 0.4875 | Val Loss: 0.7064, Val Acc: 0.4550
Epoch [3/50] | Train Loss: 0.6982, Train Acc: 0.4713 | Val Loss: 0.6940, Val Acc: 0.5400
Epoch [4/50] | Train Loss: 0.6971, Train Acc: 0.4938 | Val Loss: 0.6990, Val Acc: 0.4800
Epoch [5/50] | Train Loss: 0.6961, Train Acc: 0.4938 |

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File CSV didownload ke komputer lokal Anda.
